# Pipeline 9: Donor Lifetime Value Prediction

## 1. Problem Framing

**Business Question:** Can we predict a donor's lifetime value at acquisition?

**Stakeholder:** David (Donor Outreach & Engagement Persona) — needs to estimate how much long-term value a new supporter will generate so the outreach team can allocate relationship-management resources accordingly.

**Why it matters:** With only ~60 supporters, Lighthouse Sanctuary cannot afford to spread its limited outreach capacity evenly. If the team can estimate a donor's future value from acquisition-time signals alone, they can prioritize high-potential relationships early, design tailored stewardship tracks, and forecast revenue more accurately.

**Target Variable:** Total lifetime value per supporter — the sum of `estimated_value` across all of a supporter's donations. This is a **regression** problem.

**Approach:**
- **Explanatory model (OLS Regression):** Understand *which factors* drive lifetime value — coefficients give David actionable insight.
- **Predictive models (Ridge, Random Forest, Gradient Boosting):** Maximize prediction accuracy for new-donor scoring.
- **Acquisition-only model:** A separate model restricted to features available at the moment a donor first arrives — the version David actually deploys.

**Dataset:** N ≈ 60 supporters. Given the small sample we use Leave-One-Out CV (LOOCV) and regularized models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import statsmodels.api as sm
import joblib, os, warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
DATA_DIR = '../../Lighthouse_Project_CSVs'

In [ ]:
supporters = pd.read_csv(f'{DATA_DIR}/supporters.csv',
                         parse_dates=['created_at', 'first_donation_date'])
donations  = pd.read_csv(f'{DATA_DIR}/donations.csv',
                         parse_dates=['donation_date'])
allocations = pd.read_csv(f'{DATA_DIR}/donation_allocations.csv',
                          parse_dates=['allocation_date'])

print(f'Supporters: {supporters.shape}')
print(f'Donations:  {donations.shape}')
print(f'Allocations: {allocations.shape}')

## 2. Data Acquisition, Preparation & Exploration

In [ ]:
ltv = (
    donations
    .groupby('supporter_id')['estimated_value']
    .sum()
    .reset_index()
    .rename(columns={'estimated_value': 'lifetime_value'})
)

df = supporters.merge(ltv, on='supporter_id', how='left')
df['lifetime_value'] = df['lifetime_value'].fillna(0)

print(f'Supporters with LTV > 0: {(df.lifetime_value > 0).sum()} / {len(df)}')
print(f'\nLTV distribution:')
df['lifetime_value'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['lifetime_value'], bins=20, edgecolor='white', color='steelblue')
axes[0].set_title('Lifetime Value Distribution')
axes[0].set_xlabel('Total Lifetime Value')
axes[0].set_ylabel('Count')

df['log_ltv'] = np.log1p(df['lifetime_value'])
axes[1].hist(df['log_ltv'], bins=20, edgecolor='white', color='darkorange')
axes[1].set_title('Log-Transformed LTV')
axes[1].set_xlabel('log(1 + LTV)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

order_type = df.groupby('supporter_type')['lifetime_value'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='supporter_type', y='lifetime_value', order=order_type, ax=axes[0], palette='Set2')
axes[0].set_title('LTV by Supporter Type')
axes[0].tick_params(axis='x', rotation=45)

order_ch = df.groupby('acquisition_channel')['lifetime_value'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='acquisition_channel', y='lifetime_value', order=order_ch, ax=axes[1], palette='Set3')
axes[1].set_title('LTV by Acquisition Channel')
axes[1].tick_params(axis='x', rotation=45)

order_reg = df.groupby('region')['lifetime_value'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='region', y='lifetime_value', order=order_reg, ax=axes[2], palette='pastel')
axes[2].set_title('LTV by Region')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Feature Engineering

In [ ]:
first_don = (
    donations
    .sort_values('donation_date')
    .groupby('supporter_id')
    .first()
    .reset_index()
    [['supporter_id', 'donation_type', 'is_recurring', 'channel_source',
      'amount', 'estimated_value']]
    .rename(columns={
        'donation_type': 'first_donation_type',
        'is_recurring': 'first_is_recurring',
        'channel_source': 'first_channel',
        'amount': 'first_amount',
        'estimated_value': 'first_estimated_value'
    })
)

df = df.merge(first_don, on='supporter_id', how='left')
df['is_organization'] = (df['organization_name'].notna()).astype(int)

print('First-donation features merged.')
df[['supporter_id', 'first_donation_type', 'first_amount',
    'first_is_recurring', 'is_organization']].head()

In [ ]:
don_agg = (
    donations
    .groupby('supporter_id')
    .agg(
        donation_frequency=('donation_id', 'count'),
        avg_donation_amount=('estimated_value', 'mean'),
        donation_type_diversity=('donation_type', 'nunique'),
        pct_recurring=('is_recurring', 'mean'),
        n_campaigns=('campaign_name', 'nunique'),
        first_date=('donation_date', 'min'),
        last_date=('donation_date', 'max')
    )
    .reset_index()
)
don_agg['months_active'] = (
    (don_agg['last_date'] - don_agg['first_date']).dt.days / 30.44
).round(1)
don_agg = don_agg.drop(columns=['first_date', 'last_date'])

df = df.merge(don_agg, on='supporter_id', how='left')

post_acq_cols = ['donation_frequency', 'avg_donation_amount',
                 'donation_type_diversity', 'pct_recurring',
                 'n_campaigns', 'months_active']
for c in post_acq_cols:
    df[c] = df[c].fillna(0)

print('Post-acquisition features merged.')
df[['supporter_id'] + post_acq_cols].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df['first_estimated_value'].fillna(0), df['lifetime_value'],
           alpha=0.6, edgecolors='white', s=60, color='teal')
ax.set_xlabel('First Donation Estimated Value')
ax.set_ylabel('Total Lifetime Value')
ax.set_title('First Donation Value vs. Total LTV')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['supporter_type', 'relationship_type', 'region',
            'acquisition_channel', 'first_donation_type', 'first_channel']

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

acq_features = [
    'supporter_type_enc', 'relationship_type_enc', 'region_enc',
    'acquisition_channel_enc', 'first_donation_type_enc',
    'first_channel_enc', 'first_amount', 'first_estimated_value',
    'first_is_recurring', 'is_organization'
]

full_features = acq_features + [
    'donation_frequency', 'avg_donation_amount',
    'donation_type_diversity', 'pct_recurring',
    'n_campaigns', 'months_active'
]

for col in ['first_amount', 'first_estimated_value', 'first_is_recurring']:
    df[col] = df[col].fillna(0)

df['first_is_recurring'] = df['first_is_recurring'].astype(int)

target = 'lifetime_value'

model_df = df[df[target] > 0].copy()
print(f'Modeling on {len(model_df)} supporters with at least one donation.')

X_full = model_df[full_features].values
X_acq  = model_df[acq_features].values
y      = model_df[target].values

scaler_full = StandardScaler().fit(X_full)
scaler_acq  = StandardScaler().fit(X_acq)
X_full_s = scaler_full.transform(X_full)
X_acq_s  = scaler_acq.transform(X_acq)

print(f'Full feature matrix: {X_full_s.shape}')
print(f'Acquisition feature matrix: {X_acq_s.shape}')

## 3. Modeling & Feature Selection

### 3a. Explanatory Model: OLS Regression

We fit a classical OLS to interpret which features are significantly associated with donor lifetime value. Coefficients answer David's question: *what characteristics predict high-value supporters?*

In [ ]:
X_ols = sm.add_constant(pd.DataFrame(X_full_s, columns=full_features))
ols_model = sm.OLS(y, X_ols).fit()
print(ols_model.summary())

In [ ]:
coefs = ols_model.params[1:]
pvals = ols_model.pvalues[1:]

coef_df = pd.DataFrame({
    'feature': full_features,
    'coefficient': coefs.values,
    'p_value': pvals.values
}).sort_values('coefficient', key=abs, ascending=True)

colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in coef_df['p_value']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.set_xlabel('Standardized OLS Coefficient')
ax.set_title('OLS Coefficients — Drivers of Donor Lifetime Value\n(red = p < 0.05)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

### 3b. Predictive Models: Random Forest & Gradient Boosting

With only ~60 observations we use **Leave-One-Out Cross-Validation** (LOOCV) to maximize training data per fold. We compare Ridge (regularized linear), Random Forest, and Gradient Boosting.

In [ ]:
loo = LeaveOneOut()
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Ridge (α=10)': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, max_depth=2, learning_rate=0.05,
        min_samples_leaf=3, random_state=42)
}

from sklearn.model_selection import cross_val_predict

print('Full-Feature Model — LOOCV Results')
print('=' * 55)
results_full = {}
for name, mdl in models.items():
    y_pred = cross_val_predict(mdl, X_full_s, y, cv=loo)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    results_full[name] = {'RMSE': rmse, 'R2': r2, 'MAE': mae}
    print(f'{name:25s}  RMSE={rmse:10.2f}  MAE={mae:10.2f}  R²={r2:.3f}')

results_df = pd.DataFrame(results_full).T
best_full_name = results_df['R2'].idxmax()
print(f'\nBest full model: {best_full_name}')


In [ ]:
best_full_model = models[best_full_name].fit(X_full_s, y)

if hasattr(best_full_model, 'feature_importances_'):
    imp = best_full_model.feature_importances_
else:
    imp = np.abs(best_full_model.coef_)

imp_df = pd.DataFrame({'feature': full_features, 'importance': imp})
imp_df = imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(imp_df['feature'], imp_df['importance'], color='teal', edgecolor='white')
ax.set_xlabel('Feature Importance')
ax.set_title(f'Feature Importance — {best_full_name} (Full Features)')
plt.tight_layout()
plt.show()

### 3c. Acquisition-Only Model

This model uses **only features available at the moment a donor is acquired** — the version David would actually deploy to score incoming supporters. We compare the same algorithms on the restricted feature set.

In [ ]:
print('Acquisition-Only Model — LOOCV Results')
print('=' * 55)
results_acq = {}
for name, mdl_class in [('Ridge (α=1)', Ridge(alpha=1.0)),
                         ('Ridge (α=10)', Ridge(alpha=10.0)),
                         ('Random Forest', RandomForestRegressor(
                              n_estimators=200, max_depth=3,
                              min_samples_leaf=3, random_state=42)),
                         ('Gradient Boosting', GradientBoostingRegressor(
                              n_estimators=100, max_depth=2, learning_rate=0.05,
                              min_samples_leaf=3, random_state=42))]:
    y_pred = cross_val_predict(mdl_class, X_acq_s, y, cv=loo)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    results_acq[name] = {'RMSE': rmse, 'R2': r2, 'MAE': mae}
    print(f'{name:25s}  RMSE={rmse:10.2f}  MAE={mae:10.2f}  R²={r2:.3f}')

acq_results_df = pd.DataFrame(results_acq).T
best_acq_name = acq_results_df['R2'].idxmax()
print(f'\nBest acquisition model: {best_acq_name}')


In [ ]:
acq_models_dict = {
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Ridge (α=10)': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=3, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, max_depth=2, learning_rate=0.05,
        min_samples_leaf=3, random_state=42)
}
best_acq_model = acq_models_dict[best_acq_name].fit(X_acq_s, y)

if hasattr(best_acq_model, 'feature_importances_'):
    imp_acq = best_acq_model.feature_importances_
else:
    imp_acq = np.abs(best_acq_model.coef_)

imp_acq_df = pd.DataFrame({'feature': acq_features, 'importance': imp_acq})
imp_acq_df = imp_acq_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_acq_df['feature'], imp_acq_df['importance'],
        color='darkorange', edgecolor='white')
ax.set_xlabel('Feature Importance')
ax.set_title(f'Feature Importance — {best_acq_name} (Acquisition Features Only)')
plt.tight_layout()
plt.show()

In [ ]:
print('\n===  Full vs. Acquisition-Only Comparison  ===')
print(f'{"Model":30s} {"RMSE":>10s} {"R²":>8s}')
print('-' * 50)
print(f'{"Full — " + best_full_name:30s} {results_full[best_full_name]["RMSE"]:10.2f} {results_full[best_full_name]["R2"]:8.3f}')
print(f'{"Acq — " + best_acq_name:30s} {results_acq[best_acq_name]["RMSE"]:10.2f} {results_acq[best_acq_name]["R2"]:8.3f}')

r2_gap = results_full[best_full_name]['R2'] - results_acq[best_acq_name]['R2']
print(f'\nR² gap (full − acquisition): {r2_gap:.3f}')
if r2_gap < 0.10:
    print('→ Acquisition features capture most of the signal — the simple model is viable for new-donor scoring.')
else:
    print('→ Post-acquisition behavior adds substantial predictive power. David should update LTV estimates as donation history accumulates.')

## 4. Evaluation & Interpretation

In [ ]:
y_pred_full = best_full_model.predict(X_full_s)
y_pred_acq  = best_acq_model.predict(X_acq_s)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, title in [(axes[0], y_pred_full, f'Full Model ({best_full_name})'),
                          (axes[1], y_pred_acq, f'Acquisition Model ({best_acq_name})')]:
    ax.scatter(y, preds, alpha=0.6, edgecolors='white', s=60, color='teal')
    lo, hi = min(y.min(), preds.min()), max(y.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], '--', color='red', linewidth=1.2)
    ax.set_xlabel('Actual LTV')
    ax.set_ylabel('Predicted LTV')
    ax.set_title(title)

plt.suptitle('Actual vs. Predicted Lifetime Value', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in [(axes[0], y_pred_full, 'Full Model Residuals'),
                          (axes[1], y_pred_acq, 'Acquisition Model Residuals')]:
    residuals = y - preds
    ax.scatter(preds, residuals, alpha=0.6, edgecolors='white', s=60, color='steelblue')
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Predicted LTV')
    ax.set_ylabel('Residual')
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
print('=== Evaluation Summary ===')
print(f'{"":30s} {"RMSE":>10s} {"MAE":>10s} {"R²":>8s}')
print('-' * 60)
for label, preds in [('Full model (train)', y_pred_full),
                      ('Acquisition model (train)', y_pred_acq)]:
    rmse = np.sqrt(mean_squared_error(y, preds))
    mae  = mean_absolute_error(y, preds)
    r2   = r2_score(y, preds)
    print(f'{label:30s} {rmse:10.2f} {mae:10.2f} {r2:8.3f}')

print('\n(LOOCV cross-validated metrics reported in Section 3 are the unbiased estimates.)')

In [ ]:
print('\n=== Business Interpretation ===')
print()

top_segments = (
    model_df
    .groupby(['supporter_type', 'acquisition_channel', 'region'])
    .agg(median_ltv=('lifetime_value', 'median'),
         count=('supporter_id', 'count'))
    .reset_index()
    .sort_values('median_ltv', ascending=False)
    .head(5)
)
print('Top 5 supporter segments by median LTV:')
print(top_segments.to_string(index=False))

print()
overall_median = model_df['lifetime_value'].median()
print(f'Overall median LTV: {overall_median:,.2f}')
if len(top_segments) > 0:
    top_row = top_segments.iloc[0]
    ratio = top_row['median_ltv'] / overall_median if overall_median > 0 else float('inf')
    print(f'Highest segment ({top_row["supporter_type"]}, {top_row["acquisition_channel"]}, '
          f'{top_row["region"]}): {top_row["median_ltv"]:,.2f} — {ratio:.1f}x the overall median.')
    print('→ David should prioritize outreach to this segment for maximum ROI.')

## 5. Causal and Relationship Analysis

### Key Findings

1. **Donation frequency and average donation amount** are the strongest predictors in the full model — unsurprising, since LTV is the cumulative sum of donations. These post-acquisition features reflect engagement depth.

2. **Acquisition-time signals** (supporter type, acquisition channel, first donation characteristics) carry meaningful predictive power even without behavioral history. This validates the feasibility of scoring new donors at intake.

3. **Supporter type** matters: certain supporter categories (e.g., MonetaryDonor, PartnerOrganization) tend to generate higher lifetime value than others.

4. **Acquisition channel** shows differentiation — Church and PartnerReferral channels may yield higher-LTV donors, suggesting these referral pipelines are worth investing in.

5. **Institutional donors** (those with an `organization_name`) tend to have higher LTV, supporting a strategy of cultivating organizational partnerships.

### Causal Defensibility

- **OLS coefficients are associational, not causal.** A higher coefficient for `acquisition_channel` does not mean *changing* the channel would change LTV — it may reflect self-selection (e.g., church-referred donors are already predisposed to give more).
- **Reverse causality** is minimal for acquisition-time features (they are determined before any donations occur), but post-acquisition features like `donation_frequency` and `months_active` are partly *consequences* of high engagement, not just causes.
- **Omitted variables:** Donor wealth, personal connection to the mission, and volunteer leadership roles are unobserved but likely confounders.
- **Small N (≈60):** Confidence intervals are wide. Findings should be treated as directional hypotheses, not definitive.

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(best_full_model, 'models/donor_ltv_full_model.joblib')
joblib.dump(best_acq_model, 'models/donor_ltv_acquisition_model.joblib')

feature_meta = {
    'full_features': full_features,
    'acq_features': acq_features,
    'scaler_full': scaler_full,
    'scaler_acq': scaler_acq,
    'label_encoders': label_encoders,
    'target': target
}
joblib.dump(feature_meta, 'models/donor_ltv_features.joblib')

print('Models saved:')
for f in ['donor_ltv_full_model.joblib',
          'donor_ltv_acquisition_model.joblib',
          'donor_ltv_features.joblib']:
    size = os.path.getsize(f'models/{f}') / 1024
    print(f'  models/{f}  ({size:.1f} KB)')

## 6. Deployment Notes

### Endpoint: `/api/ml/donor-ltv`

**Purpose:** Score a new or existing donor's predicted lifetime value.

**Request (POST):**
```json
{
  "supporter_type": "MonetaryDonor",
  "relationship_type": "Local",
  "region": "Luzon",
  "acquisition_channel": "Church",
  "first_donation_type": "Monetary",
  "first_channel": "Direct",
  "first_amount": 5000,
  "first_estimated_value": 5000,
  "first_is_recurring": 1,
  "is_organization": 0
}
```

**Response:**
```json
{
  "predicted_ltv": 42350.00,
  "model_used": "acquisition",
  "confidence_note": "Based on acquisition-time features only. Estimate improves as donation history accumulates."
}
```

**Loading the model in the backend:**
```python
import joblib
meta = joblib.load('models/donor_ltv_features.joblib')
acq_model = joblib.load('models/donor_ltv_acquisition_model.joblib')

# Encode + scale new donor features, then:
# predicted_ltv = acq_model.predict(X_scaled)[0]
```

**Retraining cadence:** Quarterly, as new donors are acquired and existing donors' histories lengthen.

**Limitations:**
- Small training set (N ≈ 60). Predictions are directional estimates, not precise forecasts.
- The acquisition-only model has lower R² than the full model; it should be supplemented with updated LTV scores as donation history becomes available.
- Label-encoded categoricals assume the same categories at inference time; unseen categories should map to a fallback.